In [12]:
import pandas as pd 
import numpy as np 
import re

In [13]:
raw=pd.read_excel("project raw file.xlsx")
raw

,stNo,EmpName,Unnamed: 2
0,ST2462,Theodora Kansiime,NaN
1,ST1917,Oola Hilda,NaN
2,ST2208,Esebu Edward,NaN
3,ST2735,Muwumba Norah,NaN
4,ST1787,Kababiito Winfred,NaN
...,...,...,...
340,ST2545,Shallot Atukunda,NaN
341,ST2419,Ileka Grace,NaN
342,ST2446,Emuria Joseph,NaN
343,ST1853,Mugisa Michael,NaN


In [3]:
# Clean spaces
raw["EmpName"] = (
    raw["EmpName"]
    .astype(str)
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
)

In [14]:
# normalize spacing for display
def norm_display(s):
    return re.sub(r'\s+', ' ', str(s).strip()).title()

# choose canonical name for a series of raw names (all rows for one stNo)
def choose_canonical(names):
    s = pd.Series(names.dropna().astype(str)).str.strip().str.replace(r'\s+', ' ', regex=True)
    counts = s.value_counts()
    if counts.empty:
        return ''

    # Prefer swapped form only when swapped appears more often
    for name in counts.index:
        parts = name.split()
        if len(parts) == 2:
            swapped = f"{parts[1]} {parts[0]}"
            if swapped in counts.index and counts[swapped] > counts[name]:
                return norm_display(swapped)

    # No clear swapped preference: return most frequent normalized as First Last
    top = counts.index[0]
    parts = top.split()
    first = parts[0].title()
    last = parts[-1].title()
    return f"{first} {last}"



In [15]:
# Re-normalize EmpName and rebuild lookup (safe to re-run)
raw['EmpName'] = raw['EmpName'].astype(str).str.strip().str.replace(r'\s+',' ', regex=True)
# Re-run canonical chooser if you have it; otherwise rebuild simple most-frequent lookup:
id_lookup = raw.groupby('stNo')['EmpName'].apply(lambda x: x.value_counts().index[0]).to_dict()
raw['Standard_Name'] = raw['stNo'].map(id_lookup)

# 1) Show rows where the name was changed (if any)
changed = raw.loc[raw['EmpName'] != raw['Standard_Name'], ['stNo','EmpName','Standard_Name']]
print('Changed rows sample:', len(changed))
print(changed.head(20))

# 2) For IDs with multiple name variants, show counts and chosen canonical
for st, g in raw.groupby('stNo'):
    vc = g['EmpName'].value_counts()
    if len(vc) > 1:
        print(st, '=>', dict(vc), 'chosen:', id_lookup.get(st))

Changed rows sample: 0
Empty DataFrame
Columns: [stNo, EmpName, Standard_Name]
Index: []


In [16]:
# Usage (assumes `raw` DataFrame is loaded and EmpName cleaned)
id_lookup = raw.groupby('stNo')['EmpName'].apply(lambda x: choose_canonical(x)).to_dict()
raw['Standard_Name'] = raw['stNo'].map(id_lookup)

In [17]:
OUTPUT_FILE = "cleaned_project_file.xlsx"
raw.to_excel(OUTPUT_FILE, index=False, engine="openpyxl")
print("Saved:", OUTPUT_FILE)

Saved: cleaned_project_file.xlsx


In [ ]:
# def standardize_name(name):
#     parts = name.split()

#     # One name
#     if len(parts) == 1:
#         return parts[0].title()

#     # Two names
#     elif len(parts) == 2:
#         first, second = parts

#         # Assume surname often comes first in such cases
#         return f"{second.title()} {first.title()}"

#     # Three names
#     elif len(parts) == 3:
#         a, b, c = parts
#     #Four names
#     elif len(parts)== 4:
#         a,b,c,d =parts

#         # Put surname last
#         return f"{b.title()} {c.title()} {d.title()} {a.title()}"

#     # Four or more names
#     else:
#         surname = parts[0]
#         others = parts[1:]

#         return " ".join([x.title() for x in others] + [surname.title()])

In [ ]:
# # Apply standardization
# raw["Standard_Name"] = raw["EmpName"].apply(standardize_name)

# # Build a single standard name per ID
# id_lookup = (
#     raw.groupby("stNo")["Standard_Name"]
#       .agg(lambda x: x.value_counts().index[0])
#       .to_dict()
# )


In [ ]:
# Money cleaning function (for future files)
# def standardize_money(value):
#     if pd.isna(value):
#         return value

#     text = str(value).lower().replace(',', '').strip()
#     text = text.replace('ugx', '').strip()

#     multipliers = {
#         'k': 1_000,
#         'thousand': 1_000,
#         'm': 1_000_000,
#         'million': 1_000_000,
#         'b': 1_000_000_000,
#         'billion': 1_000_000_000,
#     }

#     match = re.match(r'(\d+(?:\.\d+)?)\s*([a-z]+)?', text)
#     if match:
#         num = float(match.group(1))
#         suffix = match.group(2)

#         if suffix in multipliers:
#             num *= multipliers[suffix]

#         return int(num)

#     return value

In [6]:
OUTPUT_FILE = "cleaned_project_file.xlsx"
# Replace all names for the same ID
raw["Standard_Name"] = raw["stNo"].map(id_lookup)

# Save output
raw.to_excel(OUTPUT_FILE, index=False, engine="openpyxl")

print(f"Saved: {OUTPUT_FILE}")

Saved: cleaned_project_file.xlsx
